In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, entropy, ttest_ind
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def district_share_variance(district_votes: torch.Tensor, S: int) -> torch.Tensor:
    eps     = 1e-8
    #max_var = 1.0 / S
    max_var = 1.0
    totals  = district_votes.sum(dim=2, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag1!")
    share   = district_votes / totals
    mean    = share.mean(dim=2, keepdim=True)
    var     = ((share - mean) ** 2).mean(dim=2)
    return (var / max_var).clamp(0.0, 1.0)

def party_share_variance(district_votes: torch.Tensor, K: int) -> torch.Tensor:
    eps = 1e-8
    #max_var = 1.0 / K
    max_var = 1.0
    totals = district_votes.sum(dim=1, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag2!")
    shares = district_votes / totals
    mean = shares.mean(dim=1, keepdim=True)
    var = ((shares - mean) ** 2).mean(dim=1)
    return (var / max_var).clamp(0.0, 1.0)

def party_seat_share(district_votes: torch.Tensor) -> torch.Tensor:
    num_examples = district_votes.shape[0]
    num_districts = district_votes.shape[1]
    num_parties = district_votes.shape[2]
    theta = [];
    for i in range(num_examples):
        seats = np.zeros(num_parties)
        for j in range(num_districts):
            # Move the tensor to CPU and convert to NumPy array before using np.argmax
            votes = district_votes[i, j, :].detach().cpu().numpy()
            k = np.argmax(votes)
            seats[k] += 1
        theta.append(seats/num_districts)
    return np.array(theta)

In [3]:
import torch.nn as nn

class EncoderNN(nn.Module):
    def __init__(self, num_districts, hidden_dims=(256, 128, 64)):
        super().__init__()
        self.input_dim = num_districts * 3

        # Deeper network with batch normalization
        layers = []
        prev = self.input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev = h

        self.mlp = nn.Sequential(*layers)

        # Separate heads for alpha and beta
        self.alpha_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

        self.beta_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, phi, voters_per_district):
        # Normalize input to [0, 1] range
        #x = phi.reshape(phi.shape[0], -1) / voters_per_district  # Use reshape instead of view
        x = phi.reshape(phi.shape[0], -1) / 1
        h = self.mlp(x)
        alpha_logits = self.alpha_head(h)
        alpha = torch.softmax(0.25*alpha_logits, dim=-1)
        beta = 0.5 + 0.5*torch.sigmoid(self.beta_head(h).squeeze(-1))
        return alpha, beta, alpha_logits

In [11]:
import os

#CKPT_DIR = "/content/drive/MyDrive/gradDPM_checkpoints"
#os.makedirs(CKPT_DIR, exist_ok=True)

def train_encoder_DPM(X, true_params, num_examples, num_districts, voters_per_district, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3,
                          gumbel_temp=0.5, verbose=True):
    torch.manual_seed(0)
    #X = torch.tensor(X_np, dtype=torch.float32, device=device)
    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    #theta_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)

    # num_district = num_districts.min()
    num_district = num_districts

    model = EncoderNN(num_districts=num_district).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()

    indices = np.arange(num_examples)
    start_time = time.time()

    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}

    best_loss = float('inf')

    for ep in range(1, epochs + 1):
        model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        indices = np.arange(num_examples)
        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                X_range.append(x)
            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)

            #phi_original = X_range
            alpha_true_b = alpha_true[batch_idx]
            beta_true_b = beta_true[batch_idx]

            # Encoder predicts alpha, beta
            alpha_pred, beta_pred, alpha_logits = model(phi_original, voters_per_district)


            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta_direct = mse_loss(beta_pred, beta_true_b.squeeze())    #predict ABM parameter
            #loss_theta_direct = mse_loss(torch.tensor(theta_reconstructed), torch.tensor(theta_true))

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss_param_direct = loss_alpha3_direct + loss_beta_direct #+ loss_theta_direct

            # 3. Combined loss with weighting
            # Start with more direct supervision, gradually shift to reconstruction
            #param_weight = max(0.1, 1.0 - ep / epochs)  # Decay from 1.0 to 0.1
            #recon_weight = 1.0 - param_weight + 0.1

            param_weight = 4.0
            recon_weight = 1.0
            variance_weight = 3.0

            loss = param_weight * loss_param_direct

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_param += loss_param_direct.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]
            #epoch_loss_theta += loss_theta_direct.item() * phi_original.shape[0]

        # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
            print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

        X_tensor_for_corr = torch.tensor(np.stack(X), dtype=torch.float32, device=device)
        alpha1_pred_all, beta1_pred_all, alpha_logits = model(X_tensor_for_corr, voters_per_district)
        #c1 = np.correlate(alpha_pred_all.cpu().numpy(), alpha_true.cpu().numpy())
        c = scipy.stats.pearsonr(beta1_pred_all.detach().cpu().numpy().squeeze(), beta_true.cpu().numpy().squeeze())[0]
        print(c)

    return model, loss_history

In [8]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_aug_DPM.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta = data['beta']
print(X.shape)
print(alpha.shape)
print(beta.shape)
x= X[0][10]
print(x.shape)

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT = 100 #int(np.sum(X[0][0][0]))
print(VOTERS_PER_DISTRICT)

train_range = range(0,2500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      Y.append((alpha[i].astype(np.float32), beta[i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta[i].astype(np.float32)))

Mounted at /content/drive
(1, 8000)
(8000, 3)
(5000, 1)
(10, 3)
100


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 50
BATCH_SIZE = 100  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_DPM, loss_history = train_encoder_DPM(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district=VOTERS_PER_DISTRICT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.75
    )

print("\nEncoder training complete.")

Device: cuda
Clearing GPU memory...
GPU memory cleared.
GPU memory summary AFTER clearing cache:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  67532 KiB |  69626 KiB |   5310 MiB |   5244 MiB |
|       from large pool |  61600 KiB |  61600 KiB |     74 MiB |     14 MiB |
|       from small pool |   5932 KiB |   8026 KiB |   5236 MiB |   5230 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  67532 KiB |  69626

In [13]:
Xtest = data['CC']

X_test_processed = []
for i in range(Xtest[0].shape[0]): # Iterate through the actual number of samples in Xtest_raw[0]
    x = Xtest[0][i].astype(np.float32)
    X_test_processed.append(x)

X_tensor = torch.tensor(np.stack(X_test_processed), dtype=torch.float32, device=device)
alpha_pred, beta_pred, _ = encoder_DPM(X_tensor, VOTERS_PER_DISTRICT)
alpha_pred = alpha_pred.detach().cpu().numpy()
beta_pred = beta_pred.detach().cpu().numpy()

print(beta[4000])
print(beta_pred[4000])
print(alpha[4000])
print(alpha_pred[4000])

print(alpha_pred[5499])
print(alpha_pred[5999])
print(alpha_pred[6499])
print(alpha_pred[6999])
print(alpha_pred[7499])
print(alpha_pred[7999])

[0.90023205]
0.8966372
[0.35850761 0.53828556 0.10320683]
[0.36243448 0.4383449  0.19922061]
[0.52258795 0.33705112 0.14036089]
[0.5552313  0.24335009 0.20141862]
[0.36034137 0.50950295 0.13015568]
[0.22565034 0.39648113 0.37786853]
[0.4287767  0.42453304 0.14669017]
[0.46692592 0.3627569  0.17031722]


Mounted at /content/drive
Processing config 0:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 1:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 2:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 3:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 4:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 5:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100

Alpha prediction for config 1, example 500: [0.4746569  0.18458608 0.340757  ]
Beta prediction for config 1, example 500: 0.9103225469589233


In [15]:
def differentiable_election_torch_batch(alpha_batch, beta_batch, num_districts, voters_per_district,
                                        device='cpu', gumbel_temp=0.5):
    """
    Improved differentiable election simulator with better gradient flow
    """
    B = alpha_batch.shape[0]
    alpha_exp = alpha_batch.unsqueeze(1).expand(-1, num_districts, -1).contiguous()
    beta_exp = beta_batch.unsqueeze(1).expand(-1, num_districts).contiguous()
    soft_counts = torch.zeros((B, num_districts, 3), device=device, dtype=torch.float32)

    for v in range(voters_per_district):
        sums = soft_counts.sum(dim=2, keepdim=True)
        # Use a small epsilon to avoid division by zero
        share = soft_counts / (sums + 1e-6)

        # Compute probabilities for this voter
        # When sum is zero, use alpha; otherwise use mixture
        probs = beta_exp.unsqueeze(2) * share + (1.0 - beta_exp).unsqueeze(2) * alpha_exp

        # Handle first voter case (when sum is 0)
        mask_zero = (sums.squeeze(2) < 0.1).unsqueeze(2)
        probs = torch.where(mask_zero, alpha_exp, probs)

        # Ensure valid probability distribution: non-negative and sums to 1
        probs = torch.relu(probs)  # Ensure non-negativity
        # Add a small epsilon to the probabilities before normalization
        # to prevent issues with zero probabilities and ensure numerical stability.
        probs = probs + 1e-10
        # Normalize to ensure the sum of probabilities for each row is exactly 1
        probs = probs / probs.sum(dim=2, keepdim=True)
        # A final clamp to ensure values are within [0, 1] range after normalization,
        # mainly to catch any extreme floating point errors.
        # probs = probs.clamp(min=0.0, max=1.0) # Removed this line as it can cause sum-to-one violations

        # Sample using Gumbel-Softmax
        probs_flat = probs.view(-1, 3)
        dist = RelaxedOneHotCategorical(temperature=gumbel_temp, probs=probs_flat)
        sample_flat = dist.rsample()
        sample = sample_flat.view(B, num_districts, 3)
        soft_counts = soft_counts + sample

    return soft_counts

In [16]:
import os

#CKPT_DIR = "/content/drive/MyDrive/gradDPM_checkpoints"
#os.makedirs(CKPT_DIR, exist_ok=True)

def train_encoder_decoder(X, true_params, num_examples, num_districts, voters_per_district_in, voters_per_district_out, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3,
                          gumbel_temp=0.5, verbose=True):
    torch.manual_seed(0)
    #X = torch.tensor(X_np, dtype=torch.float32, device=device)
    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    theta_true = torch.tensor(np.array([p[2] for p in true_params]), dtype=torch.float32, device=device)

    # num_district = num_districts.min()
    num_district = num_districts

    model = EncoderNN(num_districts=num_district).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()

    indices = np.arange(num_examples)
    start_time = time.time()

    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}

    best_loss = float('inf')

    for ep in range(1, epochs + 1):
        model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        indices = np.arange(num_examples)
        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                X_range.append(x)
            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)

            #phi_original = X_range
            alpha_true_b = alpha_true[batch_idx]
            beta_true_b = beta_true[batch_idx]
            theta_true_b = theta_true[batch_idx]

            # Encoder predicts alpha, beta
            alpha_pred, beta_pred, alpha_logits = model(phi_original, voters_per_district_in)

            # Decoder reconstructs phi
            #phi_reconstructed = differentiable_election_torch_batch(
            #    alpha_pred, beta_pred,
            #    num_districts=num_district,
            #    voters_per_district=voters_per_district,
            #    device=device,
            #    gumbel_temp=gumbel_temp
            #)

            phi_reconstructed = differentiable_election_torch_batch(
                alpha_true_b, beta_pred,
                num_districts=num_district,
                voters_per_district=voters_per_district_out,
                device=device,
                gumbel_temp=gumbel_temp
            )

            x_svar_true = district_share_variance(phi_original, num_district)
            x_kvar_true = party_share_variance(phi_original, num_parties)
            #theta_true = party_seat_share(phi_original)

            x_svar_reconstructed = district_share_variance(phi_reconstructed, num_district)
            x_kvar_reconstructed = party_share_variance(phi_reconstructed, num_parties)
            theta_reconstructed = party_seat_share(phi_reconstructed)

            # MULTI-COMPONENT LOSS:
            # 1. Reconstruction loss (MSE on normalized votes)
            loss_recon = mse_loss(phi_reconstructed / voters_per_district_out, #compare full election results
                                 phi_original / voters_per_district_in)

            loss_variance = mse_loss(x_svar_reconstructed, x_svar_true) + mse_loss(x_kvar_reconstructed, x_kvar_true)
            #loss_recon += loss_variance

            #print(beta_pred.shape)
            #print(beta_true_b.shape)

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta_direct = 2*mse_loss(beta_pred, beta_true_b.squeeze())    #predict ABM parameter
            # Fix: Ensure theta_reconstructed is converted to a tensor and moved to the correct device
            loss_theta_direct = 8*mse_loss(torch.tensor(theta_reconstructed, dtype=torch.float32).to(device), theta_true_b)

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss_param_direct = loss_alpha3_direct + loss_beta_direct + loss_theta_direct

            # 3. Combined loss with weighting
            # Start with more direct supervision, gradually shift to reconstruction
            #param_weight = max(0.1, 1.0 - ep / epochs)  # Decay from 1.0 to 0.1
            #recon_weight = 1.0 - param_weight + 0.1

            param_weight = 6.0
            recon_weight = 0
            variance_weight = 3.0

            loss = recon_weight * loss_recon + param_weight * loss_param_direct + variance_weight * loss_variance

            # Backprop
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping to prevent explosions
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_recon += loss_recon.item() * phi_original.shape[0]
            epoch_loss_variance += loss_variance.item() * phi_original.shape[0]
            epoch_loss_param += loss_param_direct.item() * phi_original.shape[0]
       #     epoch_loss_alpha += loss_alpha1_direct.item() * phi_original.shape[0]
       #     epoch_loss_alpha += loss_alpha2_direct.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]
            epoch_loss_theta += loss_theta_direct.item() * phi_original.shape[0]

        # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
            print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

        X_tensor_for_corr = torch.tensor(np.stack(X), dtype=torch.float32, device=device)
        alpha1_pred_all, beta1_pred_all, alpha_logits = model(X_tensor_for_corr, voters_per_district_in)
        #c1 = np.correlate(alpha_pred_all.cpu().numpy(), alpha_true.cpu().numpy())
        c = scipy.stats.pearsonr(beta1_pred_all.detach().cpu().numpy().squeeze(), beta_true.cpu().numpy().squeeze())[0]
        print(c)

    return model, loss_history

In [17]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_aug_DPM.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta = data['beta']
theta = data['theta']
print(X.shape)
print(alpha.shape)
print(beta.shape)
x= X[0][10]
print(x.shape)

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT_IN = 100 #int(np.sum(X[0][0][0]))
VOTERS_PER_DISTRICT_OUT = 10000 #int(np.sum(X[0][0][0]))

train_range = range(0,2500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      Y.append((alpha[i].astype(np.float32), beta[i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta[i].astype(np.float32), theta[i].astype(np.float32)))

Mounted at /content/drive
(1, 8000)
(8000, 3)
(5000, 1)
(10, 3)


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 30
BATCH_SIZE = 500  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_gradDPM, loss_history = train_encoder_decoder(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district_in=VOTERS_PER_DISTRICT_IN,
      voters_per_district_out=VOTERS_PER_DISTRICT_OUT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.25
    )

print("\nEncoder-decoder training complete.")

Device: cuda
Clearing GPU memory...
GPU memory cleared.
GPU memory summary AFTER clearing cache:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  22144 KiB | 118537 KiB |   9003 MiB |   8981 MiB |
|       from large pool |  16640 KiB | 106560 KiB |    902 MiB |    886 MiB |
|       from small pool |   5504 KiB |  15355 KiB |   8100 MiB |   8095 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  22144 KiB | 118537

In [20]:
Xtest = data['CC']

X_test_processed = []
for i in range(Xtest[0].shape[0]): # Iterate through the actual number of samples in Xtest_raw[0]
    x = Xtest[0][i].astype(np.float32)
    X_test_processed.append(x)

X_tensor = torch.tensor(np.stack(X_test_processed), dtype=torch.float32, device=device)
grad_alpha_pred, grad_beta_pred, _ = encoder_gradDPM(X_tensor, VOTERS_PER_DISTRICT)
grad_alpha_pred = grad_alpha_pred.detach().cpu().numpy()
grad_beta_pred = grad_beta_pred.detach().cpu().numpy()

print(alpha[4000])
print(beta[4000])
print(grad_alpha_pred[4000])
print(grad_beta_pred[4000])

print(grad_alpha_pred[5499])
print(grad_alpha_pred[5999])
print(grad_alpha_pred[6499])
print(grad_alpha_pred[6999])
print(grad_alpha_pred[7499])
print(grad_alpha_pred[7999])

[0.35850761 0.53828556 0.10320683]
[0.90023205]
[0.36318725 0.44332096 0.19349173]
0.86935604
[0.48985893 0.34336948 0.16677153]
[0.5591956  0.23761398 0.20319045]
[0.32871407 0.48644942 0.18483655]
[0.37854823 0.19378516 0.42766657]
[0.37431964 0.41632923 0.20935114]
[0.39535448 0.39679274 0.20785284]


In [ ]:
from scipy.io import savemat
import numpy as np
DPM_pred = {"CC1": X, "alpha1": alpha, "beta1": beta, "theta_full1": theta, "alpha_pred1": grad_alpha_pred, "beta_pred1": grad_beta_pred, "alpha_pred2": alpha_pred, "beta_pred2": beta_pred}
savemat("DPM_pred.mat", DPM_pred)

In [24]:
drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_100_10.mat'
data=sio.loadmat(mat_file_path)
X = data['CC']

# Initialize lists to store predictions for each 'j' configuration
all_alpha_preds = []
all_beta_preds = []

for j in range(0,6):

  X0 = X[0][j]
  num_examples_current = X0.shape[1]
  num_districts_current = X0[0][0].shape[0]
  num_parties_current = X0[0][0].shape[1]
  voters_per_district_current = int(np.sum(X0[0][0][0]))
  print(f"Processing config {j}:")
  print(f"  NUM_EXAMPLES: {num_examples_current}")
  print(f"  NUM_DISTRICTS: {num_districts_current}")
  print(f"  NUM_PARTIES: {num_parties_current}")
  print(f"  VOTERS_PER_DISTRICT: {voters_per_district_current}")

  train_range = range(0,500)
  num_examples_for_batch = len(train_range)

  X_np = []
  for i in range(num_examples_for_batch):
      X_np.append(X0[0][i].astype(np.float32))

  X_tensor_current = torch.tensor(np.stack(X_np), dtype=torch.float32, device=device)
  current_alpha_pred_tensor, current_beta_pred_tensor, _ = encoder_gradDPM(X_tensor_current, voters_per_district_current)

  # Convert predictions to NumPy arrays and store them in the lists
  all_alpha_preds.append(current_alpha_pred_tensor.detach().cpu().numpy())
  all_beta_preds.append(current_beta_pred_tensor.detach().cpu().numpy())

# Convert the lists of arrays into single NumPy arrays after the loop
grad_alpha_pred = np.array(all_alpha_preds)
grad_beta_pred = np.array(all_beta_preds)

print(f"\nAlpha prediction for config 1, example 500: {grad_alpha_pred[5][499]}") # Python indexing starts at 0
print(f"Beta prediction for config 1, example 500: {grad_beta_pred[5][499]}")

Mounted at /content/drive
Processing config 0:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 1:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 2:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 3:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 4:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100
Processing config 5:
  NUM_EXAMPLES: 500
  NUM_DISTRICTS: 10
  NUM_PARTIES: 3
  VOTERS_PER_DISTRICT: 100

Alpha prediction for config 1, example 500: [0.4689934  0.159406   0.37160054]
Beta prediction for config 1, example 500: 0.7793796062469482


In [21]:
drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_100_10.mat'
data=sio.loadmat(mat_file_path)
X = data['CC']

# Initialize lists to store predictions for each 'j' configuration
all_alpha_preds = []
all_beta_preds = []

for j in range(0,6):

  X0 = X[0][j]
  num_examples_current = X0.shape[1]
  num_districts_current = X0[0][0].shape[0]
  num_parties_current = X0[0][0].shape[1]
  voters_per_district_current = int(np.sum(X0[0][0][0]))
  print(f"Processing config {j}:")
  print(f"  NUM_EXAMPLES: {num_examples_current}")
  print(f"  NUM_DISTRICTS: {num_districts_current}")
  print(f"  NUM_PARTIES: {num_parties_current}")
  print(f"  VOTERS_PER_DISTRICT: {voters_per_district_current}")

  train_range = range(0,500)
  num_examples_for_batch = len(train_range)

  X_np = []
  for i in range(num_examples_for_batch):
      X_np.append(X0[0][i].astype(np.float32))

  X_tensor_current = torch.tensor(np.stack(X_np), dtype=torch.float32, device=device)
  current_alpha_pred_tensor, current_beta_pred_tensor, _ = encoder_DPM(X_tensor_current, voters_per_district_current)

  # Convert predictions to NumPy arrays and store them in the lists
  all_alpha_preds.append(current_alpha_pred_tensor.detach().cpu().numpy())
  all_beta_preds.append(current_beta_pred_tensor.detach().cpu().numpy())

# Convert the lists of arrays into single NumPy arrays after the loop
alpha_pred = np.array(all_alpha_preds)
beta_pred = np.array(all_beta_preds)

print(f"\nAlpha prediction for config 1, example 500: {alpha_pred[5][499]}") # Python indexing starts at 0
print(f"Beta prediction for config 1, example 500: {beta_pred[5][499]}")